### Pollinator Pipeline — Master Training + Inference

Trains YOLO, binary classifier, and group classifier end-to-end, then runs inference on a camera folder.
Used by the **Django backend** for scheduled retraining — calls the `pollinator` package directly.

**Input** (files needed on Drive before running) — `MyDrive/aea/`: `yolo.zip` (CVAT YOLO 1.1 export), `annotated_crops.zip` (labeled crops), `web_images.zip` (optional, iNaturalist reference), `ml_pipelines/` folder (this repo)  
**Output** — `models/yolo_best.pt`, `models/binary_best.pth`, `models/4group_insectnet.pth` · inference results in `aea/outputs/inference/{camera}/results.json`

**Must edit:**

| What | Where | Risk if skipped |
|------|-------|-----------------|
| `SMOKE_TEST = True` | Cell 2 | Set True first for a 1-epoch sanity check before committing to a full run (10–60 min) |
| `AEA_BASE` | env var or path cell | Set to your Drive root if folder is not named `aea`; on local set to repo root |

**Do not edit** any other cells — all other configs (epochs, LR, model size) have sensible defaults documented inline.


##### Run mode

`SMOKE_TEST = True` runs every training stage for 1 epoch so you can verify the pipeline end-to-end before committing to a full run.


In [ ]:
SMOKE_TEST = False


##### Install Colab deps

Only runs on Colab. Locally, install ultralytics + sahi via your project's env.


In [ ]:
# Skip this cell when running locally if ultralytics+sahi are already installed in your env.
import sys
if 'google.colab' in sys.modules:
    !pip install -q ultralytics sahi


##### 1. Train YOLO detector

Two-stage YOLO. Output: stage2/weights/best.pt.


In [ ]:
"""Train the YOLO pollinator detector on Google Colab.

Reads yolo.zip from Drive (containing data.yaml, images/{train,val,test}/,
labels/{train,val,test}/). Copies the zip to local SSD and extracts there
because Drive FUSE I/O is slow per-file across many small files.

After extracting, labels are patched:
- Lines for classes not in KEEP_CLASSES are filtered out; remaining
  indices are remapped to 0..N-1.
- Now-empty label files plus their orphan images are removed.
- data.yaml is rewritten with the KEEP_CLASSES names.

MERGE_TEST_INTO_TRAIN moves the test split into train before patching.
Use this in Colab to maximise training data. In production retraining
(via the backend's TrainingJob), the proper train/val/test split is used.

USE_TILES enables tile-based (SAHI-style) training. Source images are
sliced into overlapping TILE_SIZE x TILE_SIZE tiles before YOLO sees
them. This preserves native-resolution pixel detail on small pollinators
(30-80 px in source) that would otherwise be downsampled below the
detector's effective receptive field at imgsz=1024. IMG_SIZE auto-
aligns to TILE_SIZE in this mode so YOLO sees tiles at native scale.
The tiled dataset is cached at <extract>/yolo-tiled and reused on later
runs unless the tile config changes.

KEEP_EMPTY_TILES is a per-split dict that tells the slicer how many
background tiles to keep as negatives. The slicer subsamples in-place
during pass 2, so we never pay disk/time for tiles we will not use.
Typical setting: {'train': 2, 'val': 5, 'test': True} — tight balance
during training, full empties for the honest test number.

To re-train as an incremental fine-tune from prior weights (skip the
frozen-backbone stage), set EPOCHS_STAGE1 = 0 and point MODEL_SIZE at
the existing stage1_frozen/weights/best.pt.

Adjust the constants below to match your Drive layout, then run.
"""

import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml_pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

YOLO_ZIP_PATH = f'{BASE_DIR}/datasets/yolo.zip'
EXTRACTED_SUBDIR = 'yolo'
OUTPUT_DIR = f'{BASE_DIR}/runs/yolo'

# Class layout as exported by CVAT. Order must match the class indices in
# the label .txt files (one class per index, starting at 0). Include every
# class CVAT produces here, even ones you want to drop (like 'unsure').
CVAT_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other', 'unsure']

# Subset of CVAT_CLASSES to train on. Anything not in this list is dropped
# from labels; remaining indices are remapped to 0..N-1.
KEEP_CLASSES = ['fly', 'butterfly', 'other']

# Move test/ into train/ before training (maximum training data, val stays
# as the held-out set). Set False to keep the three-way split that the
# backend's TrainingJob retraining uses.
MERGE_TEST_INTO_TRAIN = False

# Tile-based training. When True, source images are sliced into
# overlapping TILE_SIZE x TILE_SIZE tiles and YOLO trains on the tiles.
# Use this for small-object detection on high-resolution source images.
USE_TILES = True
TILE_SIZE = 640
TILE_OVERLAP = 0.2
# Clipped bbox kept only if at least TILE_MIN_AREA of its original area
# survives the crop. 0.1 keeps boundary-clipped insects (more data); 0.3
# drops them (cleaner edges but fewer samples).
TILE_MIN_AREA = 0.1
# How many label-free tiles to keep as negatives, per split. The slicer
# samples these in-place during pass 2, so we never pay disk/time for
# tiles we will not use.
#   train: 2  -> 2 negatives per labeled tile (clean gradient signal)
#   val:   5  -> 5:1 keeps validation fast while still reflecting some
#                background dilution
#   test:  True -> keep every empty so the final test number reflects
#                  realistic inference distribution
# Set the whole thing to False to drop all negatives (degrades the
# model — useful only for ablation).
KEEP_EMPTY_TILES = {'train': 2, 'val': 5, 'test': True}

# Smoke mode: when True, both stages run 1 epoch for an end-to-end check.
SMOKE_TEST = globals().get('SMOKE_TEST', False)

# Hyperparameters. IMG_SIZE matches TILE_SIZE under tile training so
# tiles are not rescaled before being fed to YOLO.
MODEL_SIZE = 'yolo26n.pt'
IMG_SIZE = TILE_SIZE if USE_TILES else 1024
BATCH = 32
EPOCHS_STAGE1 = 1 if SMOKE_TEST else 40
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 70
LR_STAGE1 = 1e-3
LR_STAGE2 = 5e-4
SEED = 42


def ensure_extracted(zip_drive_path: str, extract_to: str, expected_subdir: str) -> str:
    """Copy zip to local SSD and unzip into <extract_to>/<expected_subdir>/."""
    import shutil
    import subprocess
    from pathlib import Path

    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)

    print(f'[setup] unzipping into {target}')
    subprocess.run(
        ['unzip', '-q', '-o', str(local_zip), '-d', str(target)],
        check=True,
    )
    local_zip.unlink()

    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()

    print(f'[setup] ready at {target}')
    return str(target)


def merge_test_into_train(dataset_root: str) -> None:
    """Move test/* into train/* for both images/ and labels/. Idempotent:
    no-op if the test split is already gone."""
    import shutil
    from pathlib import Path

    root = Path(dataset_root)
    moved = 0
    for subdir in ('images', 'labels'):
        test_dir = root / subdir / 'test'
        train_dir = root / subdir / 'train'
        if not test_dir.exists():
            continue
        train_dir.mkdir(parents=True, exist_ok=True)
        for item in list(test_dir.iterdir()):
            shutil.move(str(item), str(train_dir / item.name))
            moved += 1
        test_dir.rmdir()
    if moved:
        print(f'[merge] moved {moved} test files into train/')
    else:
        print('[merge] no test files to move (already merged or no test split)')


def _write_data_yaml(dataset_root, names) -> None:
    from pathlib import Path
    root = Path(dataset_root)
    lines = [f'path: {root}']
    for split in ('train', 'val', 'test'):
        if (root / 'images' / split).exists():
            lines.append(f'{split}: images/{split}')
    lines.append('names:')
    for i, name in enumerate(names):
        lines.append(f'  {i}: {name}')
    (root / 'data.yaml').write_text('\n'.join(lines) + '\n')
    print(f'[patch] rewrote {root / "data.yaml"} with {len(names)} classes')


def patch_dataset(
    dataset_root: str, cvat_classes: list, keep_classes: list, marker_id: str
) -> None:
    """Filter labels to keep_classes, remap indices to 0..N-1, drop now-empty
    labels and orphan images, then rewrite data.yaml.

    Idempotent via a .patched_for marker that owns the full source config
    (keep_classes + merge_test). Caller passes marker_id so the format
    stays consistent across runs.
    """
    from pathlib import Path

    root = Path(dataset_root)
    marker = root / '.patched_for'

    if marker.exists() and marker.read_text() == marker_id:
        print(f'[patch] dataset already patched for {marker_id}; skipping')
        _write_data_yaml(root, keep_classes)
        return

    cvat_to_idx = {name: i for i, name in enumerate(cvat_classes)}
    remap = {
        cvat_to_idx[name]: new_idx
        for new_idx, name in enumerate(keep_classes)
        if name in cvat_to_idx
    }

    n_lines_stripped = 0
    n_labels_removed = 0
    n_images_removed = 0

    for split in ('train', 'val', 'test'):
        labels_dir = root / 'labels' / split
        images_dir = root / 'images' / split
        if not labels_dir.exists():
            continue
        for label_file in labels_dir.glob('*.txt'):
            original = label_file.read_text().splitlines()
            kept = []
            for line in original:
                parts = line.strip().split()
                if not parts:
                    continue
                try:
                    cls = int(parts[0])
                except ValueError:
                    continue
                if cls not in remap:
                    continue
                parts[0] = str(remap[cls])
                kept.append(' '.join(parts))
            n_lines_stripped += len(original) - len(kept)
            if kept:
                label_file.write_text('\n'.join(kept) + '\n')
            else:
                label_file.unlink()
                n_labels_removed += 1
                if images_dir.exists():
                    for img in images_dir.glob(f'{label_file.stem}.*'):
                        img.unlink()
                        n_images_removed += 1

    print(
        f'[patch] keep={keep_classes}: stripped {n_lines_stripped} lines, '
        f'removed {n_labels_removed} empty labels and {n_images_removed} orphan images'
    )

    _write_data_yaml(root, keep_classes)
    marker.write_text(marker_id)


def ensure_tiled(
    source_root: str,
    extract_dir: str,
    tile_size: int,
    overlap: float,
    min_area: float,
    keep_empty,
    source_config_id: str,
    classes: list,
) -> str:
    """Slice source_root into tiles, cache result at <extract_dir>/yolo-tiled.

    Idempotent via a .sliced_for marker that captures the source patch
    state plus the tile config; mismatched config triggers a rebuild.
    """
    import shutil
    from pathlib import Path

    tiled_root = Path(extract_dir) / 'yolo-tiled'
    marker = tiled_root / '.sliced_for'
    if isinstance(keep_empty, dict):
        empty_repr = ','.join(f'{k}={v}' for k, v in sorted(keep_empty.items()))
    else:
        empty_repr = str(keep_empty)
    slice_id = (
        f'source={source_config_id}|tile={tile_size}'
        f'|overlap={overlap}|min_area={min_area}|empty={empty_repr}'
    )

    if marker.exists() and marker.read_text() == slice_id:
        print(f'[tile] reusing cached tiled dataset at {tiled_root}')
        _write_data_yaml(tiled_root, classes)
        return str(tiled_root)

    if tiled_root.exists():
        print(f'[tile] tile config changed; rebuilding {tiled_root}')
        shutil.rmtree(tiled_root)

    from pollinator.training import slice_dataset

    print(
        f'[tile] slicing {source_root} -> {tiled_root} '
        f'(tile={tile_size}, overlap={overlap}, min_area={min_area}, '
        f'empty={keep_empty})'
    )
    stats = slice_dataset(
        dataset_root=source_root,
        output_root=str(tiled_root),
        tile_size=tile_size,
        overlap=overlap,
        min_area=min_area,
        keep_empty_tiles=keep_empty,
    )
    for split, s in stats.items():
        print(
            f'[tile] {split}: {s["source_images"]} src -> '
            f'{s["tiles"]} tiles ({s["labeled_tiles"]} labeled)'
        )

    _write_data_yaml(tiled_root, classes)
    tiled_root.mkdir(parents=True, exist_ok=True)
    marker.write_text(slice_id)
    return str(tiled_root)


def main() -> None:
    import shutil
    from pathlib import Path

    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    dataset_root = ensure_extracted(
        YOLO_ZIP_PATH, LOCAL_EXTRACT_DIR, EXTRACTED_SUBDIR,
    )

    # Source config that requires a fresh extraction if it changed:
    # KEEP_CLASSES (label-remap state) and MERGE_TEST_INTO_TRAIN (test
    # split removed). patch_dataset owns the marker; main only reads it
    # to decide whether to re-extract.
    source_config_id = (
        f"keep={','.join(KEEP_CLASSES)}|merge_test={MERGE_TEST_INTO_TRAIN}"
    )
    marker = Path(dataset_root) / '.patched_for'
    if marker.exists() and marker.read_text() != source_config_id:
        print('[setup] config changed since last run; re-extracting from zip...')
        shutil.rmtree(dataset_root)
        dataset_root = ensure_extracted(
            YOLO_ZIP_PATH, LOCAL_EXTRACT_DIR, EXTRACTED_SUBDIR,
        )

    if MERGE_TEST_INTO_TRAIN:
        merge_test_into_train(dataset_root)

    patch_dataset(
        dataset_root,
        cvat_classes=CVAT_CLASSES,
        keep_classes=KEEP_CLASSES,
        marker_id=source_config_id,
    )

    if USE_TILES:
        dataset_root = ensure_tiled(
            source_root=dataset_root,
            extract_dir=LOCAL_EXTRACT_DIR,
            tile_size=TILE_SIZE,
            overlap=TILE_OVERLAP,
            min_area=TILE_MIN_AREA,
            keep_empty=KEEP_EMPTY_TILES,
            source_config_id=source_config_id,
            classes=KEEP_CLASSES,
        )

    from pollinator.workflows.training_yolo import retrain_yolo

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] {processed}/{total} {message}')

    result = retrain_yolo(
        use_tiles=False,
        dataset_root=dataset_root,
        output_dir=OUTPUT_DIR,
        model_size=MODEL_SIZE,
        img_size=IMG_SIZE,
        batch=BATCH,
        epochs_stage1=EPOCHS_STAGE1,
        epochs_stage2=EPOCHS_STAGE2,
        lr_stage1=LR_STAGE1,
        lr_stage2=LR_STAGE2,
        classes=KEEP_CLASSES,
        seed=SEED,
        progress_callback=on_progress,
    )

    print('\n=== Training complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()


##### 2. Train EfficientNet binary classifier

Output: binary_best.pth.


In [ ]:
"""Train the EfficientNet binary insect/background classifier on Colab.

Reads `annotated_crops.zip` from Drive (containing labeled_ls/ and
labeled_mb/, each with bumblebee, fly, butterfly, other, background
subdirs). Copies the zip to local SSD and extracts there because Drive
FUSE I/O is slow per-file; one big copy plus a local unzip is dramatically
faster than reading the unzipped tree off Drive.

Adjust the constants below to match your Drive layout, then run.
"""

# Environment detection and paths.
# In Colab: defaults to MyDrive/aea/ and /content/data for extraction.
# Local: set AEA_BASE env var (or accept ~/aea default). PIPELINE_ROOT
# defaults to <BASE>/ml_pipelines; override with AEA_PIPELINE_ROOT if
# your repo lives elsewhere.
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml_pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

ANNOTATED_ZIP_PATH = f'{BASE_DIR}/datasets/annotated_crops.zip'
OUTPUT_DIR = f'{BASE_DIR}/runs/binary'

# Model. 'efficientnet' uses torchvision-pretrained EfficientNet-B2.
# 'insectnet' loads INSECTNET_WEIGHTS as a backbone init.
MODEL_TYPE = 'efficientnet'
INSECTNET_WEIGHTS = f'{BASE_DIR}/weights/insectnet_pretrained.pth'

# Smoke mode: see train_yolo.py for the full explanation. When True,
# epochs collapse to 1 for a fast end-to-end sanity check.
SMOKE_TEST = globals().get('SMOKE_TEST', False)

# Hyperparameters.
EPOCHS = 1 if SMOKE_TEST else 20
BATCH = 32
LR = 1e-3
VAL_FRAC = 0.2
TEST_FRAC = 0.0
BG_RATIO = 3
SEED = 42

def ensure_extracted(zip_drive_path: str, extract_to: str, expected_subdir: str) -> str:
    """Copy zip from Drive to local SSD and unzip into <extract_to>/<expected_subdir>/.

    Handles zips with or without a top-level wrapping directory. If the zip
    wraps its content in a same-named dir, the contents are flattened up one
    level so the returned path always contains the actual data.
    """
    import shutil
    import subprocess
    from pathlib import Path

    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)

    print(f'[setup] unzipping into {target}')
    subprocess.run(
        ['unzip', '-q', '-o', str(local_zip), '-d', str(target)],
        check=True,
    )
    local_zip.unlink()

    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()

    print(f'[setup] ready at {target}')
    return str(target)


def main() -> None:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    annotated_root = ensure_extracted(
        ANNOTATED_ZIP_PATH, LOCAL_EXTRACT_DIR, 'annotated_crops',
    )
    # Auto-discover all labeled dataset folders (skip 'progress').
    # Add new folders to annotated_crops.zip and they are picked up automatically.
    import os as _os
    data_dirs = sorted(
        f'{annotated_root}/{d}' for d in _os.listdir(annotated_root)
        if _os.path.isdir(f'{annotated_root}/{d}') and d != 'progress'
    )
    print(f'Datasets: {[_os.path.basename(d) for d in data_dirs]}')

    from pollinator.workflows.training_binary import retrain_binary

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] epoch {processed}/{total} {message}')

    result = retrain_binary(
        data_dirs=data_dirs,
        model_type=MODEL_TYPE,
        insectnet_weights=INSECTNET_WEIGHTS if MODEL_TYPE == 'insectnet' else None,
        output_dir=OUTPUT_DIR,
        epochs=EPOCHS,
        batch=BATCH,
        lr=LR,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
        bg_ratio=BG_RATIO,
        seed=SEED,
        progress_callback=on_progress,
    )

    print('\n=== Training complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()


##### 3. Train InsectNet group classifier

Output: group_best.pth.


In [ ]:
"""Train the group classifier (bumblebee/fly/butterfly/other) on Colab.

Reuses `annotated_crops.zip` (the four insect subdirs; background is
ignored) and adds `web_images.zip` for iNaturalist-style augmentation.

Two stages: stage 1 trains the head, stage 2 (optional) unfreezes the
last block. EPOCHS_S2=0 is recommended for InsectNet on small Arctic
data since stage 2 tends to overfit.

Adjust the constants below to match your Drive layout, then run.
"""

# Environment detection and paths.
# In Colab: defaults to MyDrive/aea/ and /content/data for extraction.
# Local: set AEA_BASE env var (or accept ~/aea default). PIPELINE_ROOT
# defaults to <BASE>/ml_pipelines; override with AEA_PIPELINE_ROOT if
# your repo lives elsewhere.
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml_pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

ANNOTATED_ZIP_PATH = f'{BASE_DIR}/datasets/annotated_crops.zip'
WEB_ZIP_PATH = f'{BASE_DIR}/datasets/web_images.zip'
OUTPUT_DIR = f'{BASE_DIR}/runs/group'

# Model. 'insectnet' needs INSECTNET_WEIGHTS; 'efficientnet' does not.
MODEL_TYPE = 'insectnet'
INSECTNET_WEIGHTS = f'{BASE_DIR}/weights/insectnet_pretrained.pth'

# Smoke mode: see train_yolo.py for the full explanation. When True,
# epochs collapse to 1 for a fast end-to-end sanity check.
SMOKE_TEST = globals().get('SMOKE_TEST', False)

# Hyperparameters.
EPOCHS_S1 = 1 if SMOKE_TEST else 20
EPOCHS_S2 = 0
BATCH = 32
LR_S1 = 1e-3
LR_S2 = 1e-4
VAL_FRAC = 0.2
TEST_FRAC = 0.0
SEED = 42

def ensure_extracted(zip_drive_path: str, extract_to: str, expected_subdir: str) -> str:
    """Copy zip from Drive to local SSD and unzip into <extract_to>/<expected_subdir>/.

    Handles zips with or without a top-level wrapping directory. If the zip
    wraps its content in a same-named dir, the contents are flattened up one
    level so the returned path always contains the actual data.
    """
    import shutil
    import subprocess
    from pathlib import Path

    target = Path(extract_to) / expected_subdir
    if target.exists() and any(target.iterdir()):
        print(f'[setup] reusing existing {target}')
        return str(target)

    target.mkdir(parents=True, exist_ok=True)
    local_zip = target.parent / Path(zip_drive_path).name
    print(f'[setup] copying {zip_drive_path} -> {local_zip}')
    shutil.copy(zip_drive_path, local_zip)

    print(f'[setup] unzipping into {target}')
    subprocess.run(
        ['unzip', '-q', '-o', str(local_zip), '-d', str(target)],
        check=True,
    )
    local_zip.unlink()

    nested = target / expected_subdir
    if nested.is_dir():
        for item in list(nested.iterdir()):
            shutil.move(str(item), str(target / item.name))
        nested.rmdir()

    print(f'[setup] ready at {target}')
    return str(target)


def main() -> None:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    annotated_root = ensure_extracted(
        ANNOTATED_ZIP_PATH, LOCAL_EXTRACT_DIR, 'annotated_crops',
    )
    web_root = ensure_extracted(
        WEB_ZIP_PATH, LOCAL_EXTRACT_DIR, 'web_images',
    )
    # Auto-discover all labeled dataset folders (skip 'progress').
    # Add new folders to annotated_crops.zip and they are picked up automatically.
    import os as _os
    data_dirs = sorted(
        f'{annotated_root}/{d}' for d in _os.listdir(annotated_root)
        if _os.path.isdir(f'{annotated_root}/{d}') and d != 'progress'
    )
    print(f'Datasets: {[_os.path.basename(d) for d in data_dirs]}')
    web_dir = web_root

    from pollinator.workflows.training_group import retrain_group

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] epoch {processed}/{total} {message}')

    result = retrain_group(
        data_dirs=data_dirs,
        web_dir=web_dir,
        model_type=MODEL_TYPE,
        insectnet_weights=INSECTNET_WEIGHTS if MODEL_TYPE == 'insectnet' else None,
        output_dir=OUTPUT_DIR,
        epochs_s1=EPOCHS_S1,
        epochs_s2=EPOCHS_S2,
        batch=BATCH,
        lr_s1=LR_S1,
        lr_s2=LR_S2,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
        seed=SEED,
        progress_callback=on_progress,
    )

    print('\n=== Training complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()


##### 4. Run inference on a camera folder

Output: results.json.


In [ ]:
"""Run the full pollinator inference pipeline on Colab.

Processes one camera-plot folder end-to-end: YOLO detection in parallel
with classical preprocessing, then InsectNet binary gate, then group
classifier, then merge by IoU. Writes results.json plus per-detection
crops under OUTPUT_DIR.

Adjust the constants below to match your Drive layout, then run.
"""

# Environment detection and paths.
# In Colab: defaults to MyDrive/aea/ and /content/data for extraction.
# Local: set AEA_BASE env var (or accept ~/aea default). PIPELINE_ROOT
# defaults to <BASE>/ml_pipelines; override with AEA_PIPELINE_ROOT if
# your repo lives elsewhere.
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = os.environ.get(
    'AEA_BASE',
    '/content/drive/MyDrive/aea' if IN_COLAB else os.path.expanduser('~/aea'),
)
PIPELINE_ROOT = os.environ.get('AEA_PIPELINE_ROOT', f'{BASE_DIR}/ml_pipelines')
LOCAL_EXTRACT_DIR = os.environ.get(
    'AEA_EXTRACT',
    '/content/data' if IN_COLAB else f'{BASE_DIR}/extracted',
)

IMAGE_DIR = f'{BASE_DIR}/inference/camera_a'
OUTPUT_DIR = f'{BASE_DIR}/runs/inference/camera_a'

# Model checkpoints.
YOLO_MODEL = f'{BASE_DIR}/runs/yolo/stage2_finetune/weights/best.pt'
BINARY_MODEL = f'{BASE_DIR}/runs/binary/efficientnet_binary_best.pth'
GROUP_MODEL = f'{BASE_DIR}/runs/group/group_insectnet_best.pth'

# Knobs that the upload UI also exposes. Defaults match the production
# pipeline; override per-camera if needed.
YOLO_CONFIDENCE = 0.4
BINARY_THRESHOLD = 0.5
IOU_THRESHOLD = 0.3
SKIP_FIRST_N = 0
DEBUG = False

PREPROCESSING_CONFIG = {
    'crop_pad_frac': 0.3,
    'background_sample_size': 100,
    'min_contour_area': 400,
    'max_contour_area': 35000,
    'sunny_shutter_threshold': 150,
    'skip_flash': True,
    'skip_foggy': True,
    'enable_large_motion': True,
}

def main() -> None:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    if PIPELINE_ROOT and PIPELINE_ROOT not in sys.path:
        sys.path.insert(0, PIPELINE_ROOT)

    from pollinator.workflows.inference import run_pipeline

    def on_progress(processed: int, total: int, message: str = '', level: str = 'info') -> None:
        if message:
            print(f'[{level}] {processed}/{total} {message}')

    result = run_pipeline(
        image_dir=IMAGE_DIR,
        output_dir=OUTPUT_DIR,
        yolo_model=YOLO_MODEL,
        binary_model=BINARY_MODEL,
        group_model=GROUP_MODEL,
        config=PREPROCESSING_CONFIG,
        yolo_confidence=YOLO_CONFIDENCE,
        binary_threshold=BINARY_THRESHOLD,
        iou_threshold=IOU_THRESHOLD,
        skip_first_n=SKIP_FIRST_N,
        debug=DEBUG,
        progress_callback=on_progress,
    )

    print('\n=== Inference complete ===')
    for k, v in result.items():
        print(f'{k}: {v}')


if __name__ == '__main__':
    main()
